<a href="https://colab.research.google.com/github/sanaisrail/urdu-ocr-codesaviours-si26-Sana/blob/main/Week4_Urdu_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers==4.46.3 sentencepiece datasets accelerate

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset (1).csv")

print(df.head())
print(df.columns)
print("Total Samples:", len(df))

                                      image  \
0   /content/drive/MyDrive/data/other/1.png   
1  /content/drive/MyDrive/data/other/20.png   
2   /content/drive/MyDrive/data/other/5.png   
3  /content/drive/MyDrive/data/other/35.png   
4  /content/drive/MyDrive/data/other/34.png   

                                                text  
0  ایران کے گلستان صوبے میں واقع آقتکہ خان ریلوے ...  
1  روان سال فروری میں، ایران کے خلاف امریکہ اور ا...  
2  اس پل کی اہمیت کو دو اہم بین الاقوامی راستوں ک...  
3  ایرانی ریلوے حکام کے مطابق، گذشتہ سال کم از کم...  
4         پٹرول سے سیرامکس تک تجارت کا ایک اہم راستہ  
Index(['image', 'text'], dtype='object')
Total Samples: 200


In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

print("✅ Model loaded successfully")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "pa

✅ Model loaded successfully


In [ ]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Decoder Start Token:", model.config.decoder_start_token_id)
print("Pad Token:", model.config.pad_token_id)
print("EOS Token:", model.config.eos_token_id)

Decoder Start Token: 0
Pad Token: 1
EOS Token: 2


In [ ]:
import pandas as pd
import cv2
import numpy as np

from PIL import Image
from torch.utils.data import Dataset

class UrduOCRDataset(Dataset):

    def __init__(self, csv_file, processor):
        self.data = pd.read_csv(csv_file)
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Read image
        image = cv2.imread(row["image"])

        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # OTSU Thresholding
        _, thresh = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        # Resize
        thresh = cv2.resize(thresh, (384, 384))

        # Convert back to RGB
        image = Image.fromarray(thresh).convert("RGB")

        # Image preprocessing
        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Text preprocessing
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        # Ignore padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [ ]:
dataset = UrduOCRDataset(
    "/content/drive/MyDrive/dataset (1).csv",
    processor
)

print("Dataset Loaded:", len(dataset))

Dataset Loaded: 200


In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training Samples:", len(train_dataset))
print("Testing Samples:", len(test_dataset))

Training Samples: 160
Testing Samples: 40


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 40
Testing batches: 10


In [ ]:
from transformers import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Optimizer Ready")

Optimizer Ready


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Decoder Start:", model.config.decoder_start_token_id)
print("Pad:", model.config.pad_token_id)
print("EOS:", model.config.eos_token_id)

Decoder Start: 0
Pad: 1
EOS: 2


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("\n✅ Training Complete!")


Epoch 1/10
------------------------------
Batch 0/40 | Loss: 17.4786
Batch 10/40 | Loss: 5.4220
Batch 20/40 | Loss: 4.2846
Batch 30/40 | Loss: 3.8196
Epoch 1 Average Loss: 5.1538

Epoch 2/10
------------------------------
Batch 0/40 | Loss: 3.6676
Batch 10/40 | Loss: 3.6255
Batch 20/40 | Loss: 3.5848
Batch 30/40 | Loss: 3.3792
Epoch 2 Average Loss: 3.6067

Epoch 3/10
------------------------------
Batch 0/40 | Loss: 3.5203
Batch 10/40 | Loss: 3.5463
Batch 20/40 | Loss: 3.5009
Batch 30/40 | Loss: 3.4731
Epoch 3 Average Loss: 3.5415

Epoch 4/10
------------------------------
Batch 0/40 | Loss: 3.4552
Batch 10/40 | Loss: 3.6614
Batch 20/40 | Loss: 3.5612
Batch 30/40 | Loss: 3.4383
Epoch 4 Average Loss: 3.5159

Epoch 5/10
------------------------------
Batch 0/40 | Loss: 3.5295
Batch 10/40 | Loss: 3.4469
Batch 20/40 | Loss: 3.3408
Batch 30/40 | Loss: 3.5716
Epoch 5 Average Loss: 3.4853

Epoch 6/10
------------------------------
Batch 0/40 | Loss: 3.4589
Batch 10/40 | Loss: 3.5259
Batch 20

In [ ]:
generated_ids = model.generate(
    pixel_values,
    max_new_tokens=128,
    num_beams=4,
    early_stopping=True
)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        predictions = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels = torch.where(
            labels == -100,
            processor.tokenizer.pad_token_id,
            labels
        )

        actuals = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(predictions, actuals):

            print("Predicted:", pred)
            print("Actual   :", actual)
            print("-" * 50)

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

accuracy = (correct / total) * 100

print(f"\nAccuracy: {accuracy:.2f}%")

Predicted: ������ � ����ا���ری� �ن�تے�� �� �ا �ر �� ، �  �ن �ت ؒ ����ا�ر��ڌ� �نڒ�ت�ا��وہ�ا� �و ؁ �
Actual   : مظاہرین نے مصروف شاہراہ بلاک کردی اور خالی کرنے سے
--------------------------------------------------
Predicted: ��ا�� �ی��ا � �اا� ��ڌا� ،����ا� ���ر�رار �ر�ت�تات �تځہا� ؁� � ا  � �و�واو �و�ن�نان �ن�ا��ا�اا� �ا�ک۩ا� ة�م�مام �م�ل�لال �ل� �� �ا � � �ڒےا� �
Actual   : پیٹرول ختم ہونے کا اشارہ دینے والی لائٹ روشن ہونے پر گاڑی کتنا فاصلہ طے کر سکتی ہے؟
--------------------------------------------------
Predicted: �ح�تی���� ���ڌ��� ����ا� �ت� � ،،ا�ت�ا�ت �� �ا � � �  �ت�اا� ا ة�ر۩��ک ةات  تتا� ��ر�وہ�ا��وځ�ر�ر �و�و ؁ �رار ر؁�ا��مے�ن�مڒ�ا� �
Actual   : پاک انسٹی ٹیوٹ فار پیس اسٹڈیز کی 'پاکستان سیکیورٹی رپورٹ 2025' کے مطابق ملک بھر میں 699 دہشت گرد حملے �
--------------------------------------------------
Predicted: مح����� ��� �یڌ ،�ر�ر��� � �� � �ر �  �،�ر� ��  ر �رر��ا�ا�ا �ا�ا اراة۩ک ة�� �ر��و�و�و �و�و ورو�ن�ن�ن �ن�ن نرن�ت�ت�ت �ت�ت ترت�
Actual   : کے گرڈ اسٹیشن پر حملہ کیا۔
------

In [ ]:
print(model.config.decoder_start_token_id)
print(model.config.pad_token_id)
print(model.config.eos_token_id)

0
1
2


In [ ]:
print(processor.tokenizer.decode(labels[0][labels[0] != -100]))

<s>صبر اور شکر کامیابی کا راز ہیں</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [ ]:
sample = dataset[0]

pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

generated_ids = model.generate(
    pixel_values,
    max_new_tokens=128,
    num_beams=4
)

print("Prediction:")
print(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])

print("\nGround Truth:")
print(processor.tokenizer.decode(
    sample["labels"][sample["labels"] != -100]
))

Prediction:
٭�����������������������������������������������������������������������������������������������������������������������������

Ground Truth:
<s>ایران کے گلستان صوبے میں واقع آقتکہ خان ریلوے پل پر امریکی حملے نے ایران کے اہم ترین ٹرانزٹ راستوں می�</s>


In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

In [ ]:
model.config.decoder_start_token_id = 0

In [ ]:
print(model.config.decoder_start_token_id)
0

0


0